# Scenario 3 — ANN-based 2D Navigation (local)

Notebook này đọc file kết quả `results\\Scenario_3_Novel_Final.csv` (tạo bởi `Scenario_3/new.py`) và tái tạo các thống kê/biểu đồ theo tinh thần paper: **effectiveness (NER, NoVS)** và **efficiency (EtTQ)**.

**Scenario 3**: Tối ưu bộ điều khiển ANN cho robot differential-drive điều hướng 2D.
- 3 arena: Small, Large, Maze
- 3 cấu hình cảm biến: m ∈ {3, 5, 9} (proximity sensors)
- p ∈ {122, 212, 464} tham số ANN
- Fitness: khoảng cách trung bình robot–target trong 60s (minimize)

In [ ]:
# ── Cài thư viện nếu chưa có ──────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'pandas', 'numpy', 'matplotlib', 'scipy', 'openpyxl',
                '--quiet'], check=False)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

# ── Cấu hình ──────────────────────────────────────────────────
experiment_name = 'Scenario_3'
file_name = 'results\\Scenario_3_Novel_Final.csv'
separator = ';'
delta_quantization = 100
level_significance = 0.01
efficiency_percentile_reference = 0.75

# ── Tạo thư mục output ────────────────────────────────────────
for folder in [f'Figures/{experiment_name}',
               f'LaTeX/{experiment_name}',
               f'XLSX Files/{experiment_name}']:
    os.makedirs(folder, exist_ok=True)

print('Thư mục output đã sẵn sàng.')

In [ ]:
# ── Đọc CSV kết quả ───────────────────────────────────────────
df_raw = pd.read_csv(file_name, sep=separator, low_memory=False)
print(f'Đọc xong: {len(df_raw):,} dòng')
df_raw.head(3)

In [ ]:
# ── Tiền xử lý cột ────────────────────────────────────────────
df = df_raw.copy()

# Đổi tên cột solver
if 'solver_sigma' in df.columns:
    df.rename(columns={'solver_sigma': 'solver'}, inplace=True)

# Đảm bảo kiểu số
df['evals']        = pd.to_numeric(df['evals'],        errors='coerce')
df['best_fitness'] = pd.to_numeric(df['best_fitness'], errors='coerce')
df['seed']         = pd.to_numeric(df['seed'],         errors='coerce')

# Lấy genotype_size mỗi problem
if 'genotype_size' in df.columns:
    df['genotype_size'] = pd.to_numeric(df['genotype_size'], errors='coerce')

# Danh sách solvers và problems
solvers  = sorted(df['solver'].unique())
problems = sorted(df['problem'].unique())
print(f'Solvers  ({len(solvers)}): {solvers}')
print(f'Problems ({len(problems)}): {problems}')

In [ ]:
# ── Convergence plot ──────────────────────────────────────────
# Với mỗi problem: vẽ median best_fitness theo evals cho từng solver

n_problems = len(problems)
ncols = 3
nrows = (n_problems + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows),
                          constrained_layout=True)
axes = np.array(axes).reshape(-1)

cmap   = mpl.colormaps.get_cmap('tab10')
colors = {s: cmap(i / max(len(solvers) - 1, 1)) for i, s in enumerate(solvers)}

for ax, prob in zip(axes, problems):
    sub = df[df['problem'] == prob]
    # Quantise evals
    sub = sub.copy()
    sub['evals_q'] = (sub['evals'] // delta_quantization) * delta_quantization
    for solver in solvers:
        ss = sub[sub['solver'] == solver]
        grp = ss.groupby('evals_q')['best_fitness'].median().reset_index()
        grp = grp.sort_values('evals_q')
        ax.plot(grp['evals_q'], grp['best_fitness'],
                label=solver, color=colors[solver], linewidth=1.2)
    ax.set_title(prob, fontsize=9)
    ax.set_xlabel('Evals')
    ax.set_ylabel('Median best fitness')
    ax.grid(True, alpha=0.3)

# Ẩn axes dư
for ax in axes[n_problems:]:
    ax.set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center',
           ncol=min(len(solvers), 5), fontsize=8,
           bbox_to_anchor=(0.5, -0.02))
fig.suptitle(f'{experiment_name} — Convergence (median best fitness)', fontsize=13)

conv_path = f'Figures/{experiment_name}/{experiment_name}_Convergence.png'
fig.savefig(conv_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {conv_path}')

In [ ]:
# ── NER — Normalized Effectiveness Rank ───────────────────────
# Với mỗi (problem, seed): lấy best_fitness cuối cùng của mỗi solver,
# rank (1 = tốt nhất = nhỏ nhất vì minimize), NER = 1 - (rank-1)/(n_solvers-1)

# Lấy dòng cuối (evals lớn nhất) mỗi (problem, solver, seed)
final = (
    df.sort_values('evals')
      .groupby(['problem', 'solver', 'seed'], as_index=False)
      .last()
)

# Tính rank trong mỗi (problem, seed)
final['rank'] = final.groupby(['problem', 'seed'])['best_fitness'] \
                     .rank(method='average', ascending=True)   # minimize → nhỏ hơn = rank thấp hơn = tốt hơn

n_solvers = len(solvers)
final['NER'] = 1.0 - (final['rank'] - 1) / max(n_solvers - 1, 1)

ner_mean = final.groupby(['problem', 'solver'])['NER'].mean().reset_index()
ner_mean.rename(columns={'NER': 'mean_NER'}, inplace=True)
print('NER sample:')
ner_mean.head()

In [ ]:
# ── NER bar chart ─────────────────────────────────────────────
ner_pivot = ner_mean.pivot(index='problem', columns='solver', values='mean_NER')

fig, ax = plt.subplots(figsize=(max(8, len(problems) * 0.9), 5))
ner_pivot.plot(kind='bar', ax=ax, colormap='tab10', edgecolor='white', linewidth=0.5)
ax.set_title(f'{experiment_name} — Mean NER per Problem', fontsize=12)
ax.set_xlabel('Problem')
ax.set_ylabel('Mean NER')
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, label='chance')
ax.legend(title='Solver', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.3)

ner_path = f'Figures/{experiment_name}/{experiment_name}_NER.png'
fig.savefig(ner_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {ner_path}')

In [ ]:
# ── EtTQ — Evals to Third Quartile ───────────────────────────
# Q3 của final best fitness (qua toàn bộ EA × runs × problems)
# EtTQ = eval đầu tiên mà best_fitness <= threshold

q3_threshold = final['best_fitness'].quantile(efficiency_percentile_reference)
print(f'Q3 threshold (p={efficiency_percentile_reference}): {q3_threshold:.6f}')

ettq_rows = []
for prob in problems:
    for solver in solvers:
        for seed_val in df['seed'].unique():
            sub = df[(df['problem'] == prob) &
                     (df['solver']  == solver) &
                     (df['seed']    == seed_val)].sort_values('evals')
            if sub.empty:
                continue
            reached = sub[sub['best_fitness'] <= q3_threshold]
            ettq    = int(reached['evals'].iloc[0]) if not reached.empty else np.nan
            ettq_rows.append({'problem': prob, 'solver': solver,
                               'seed': seed_val, 'EtTQ': ettq})

ettq_df = pd.DataFrame(ettq_rows)
ettq_mean = ettq_df.groupby(['problem', 'solver'])['EtTQ'].mean().reset_index()
ettq_mean.rename(columns={'EtTQ': 'mean_EtTQ'}, inplace=True)
print(ettq_mean.head())

In [ ]:
# ── EtTQ bar chart ────────────────────────────────────────────
ettq_pivot = ettq_mean.pivot(index='problem', columns='solver', values='mean_EtTQ')

fig, ax = plt.subplots(figsize=(max(8, len(problems) * 0.9), 5))
ettq_pivot.plot(kind='bar', ax=ax, colormap='tab10', edgecolor='white', linewidth=0.5)
ax.set_title(f'{experiment_name} — Mean EtTQ (p={efficiency_percentile_reference})', fontsize=12)
ax.set_xlabel('Problem')
ax.set_ylabel('Mean evals to Q3 threshold')
ax.legend(title='Solver', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.3)

ettq_path = f'Figures/{experiment_name}/{experiment_name}_Efficiency_percentile-{efficiency_percentile_reference}.png'
fig.savefig(ettq_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {ettq_path}')

In [ ]:
# ── Mean NER vs. p (genotype size) ───────────────────────────
if 'genotype_size' in df.columns:
    geno_map = final.groupby('problem')['genotype_size'].median().to_dict()

    ner_p = ner_mean.copy()
    ner_p['p'] = ner_p['problem'].map(geno_map)
    ner_p = ner_p.dropna(subset=['p'])

    fig, ax = plt.subplots(figsize=(8, 5))
    for solver in solvers:
        sub = ner_p[ner_p['solver'] == solver].sort_values('p')
        ax.plot(sub['p'], sub['mean_NER'], marker='o', label=solver,
                color=colors[solver], linewidth=1.5)
    ax.set_title(f'{experiment_name} — Mean NER vs. genotype size p', fontsize=12)
    ax.set_xlabel('p (genotype size)')
    ax.set_ylabel('Mean NER')
    ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8)
    ax.legend(title='Solver', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
    ax.grid(alpha=0.3)

    ner_p_path = f'Figures/{experiment_name}/{experiment_name}_meanNER-vs-p.png'
    fig.savefig(ner_p_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {ner_p_path}')
else:
    print('Cột genotype_size không có trong dữ liệu, bỏ qua biểu đồ NER vs p.')

## 3) Efficiency: EtTQ

EtTQ = số evals để đạt ngưỡng $Q_3$ của phân phối final best fitness (gộp tất cả EA×runs trên cùng problem).

Solver nào có EtTQ **nhỏ hơn** đạt ngưỡng **nhanh hơn** → hiệu quả hơn.

In [ ]:
# ── NoVS — Number of Victories Score ─────────────────────────
# Với mỗi cặp solver (A, B) trên mỗi problem: Wilcoxon test (final best_fitness)
# NoVS(A) = số lần A thắng B có ý nghĩa thống kê

import itertools

novs_records = []

for prob in problems:
    sub_prob = final[final['problem'] == prob]
    for s_a, s_b in itertools.combinations(solvers, 2):
        vals_a = sub_prob[sub_prob['solver'] == s_a]['best_fitness'].dropna().values
        vals_b = sub_prob[sub_prob['solver'] == s_b]['best_fitness'].dropna().values
        n = min(len(vals_a), len(vals_b))
        if n < 5:
            continue
        vals_a, vals_b = vals_a[:n], vals_b[:n]
        try:
            stat, pval = wilcoxon(vals_a, vals_b, alternative='two-sided')
        except ValueError:
            pval = 1.0
        mean_a = vals_a.mean()
        mean_b = vals_b.mean()
        if pval < level_significance:
            if mean_a < mean_b:   # minimize: a thắng
                novs_records.append({'problem': prob, 'winner': s_a, 'loser': s_b, 'p_value': pval})
            else:
                novs_records.append({'problem': prob, 'winner': s_b, 'loser': s_a, 'p_value': pval})

novs_df   = pd.DataFrame(novs_records) if novs_records else pd.DataFrame(columns=['problem','winner','loser','p_value'])
novs_cnt  = novs_df.groupby('winner').size().reindex(solvers, fill_value=0).rename('NoVS')
print('NoVS:')
print(novs_cnt.sort_values(ascending=False))

In [ ]:
# ── NoVS per problem ─────────────────────────────────────────
novs_per_prob = (
    novs_df.groupby(['problem', 'winner']).size()
           .unstack(fill_value=0)
           .reindex(columns=solvers, fill_value=0)
) if not novs_df.empty else pd.DataFrame(0, index=problems, columns=solvers)

fig, ax = plt.subplots(figsize=(max(8, len(problems) * 0.9), 5))
novs_per_prob.plot(kind='bar', ax=ax, colormap='tab10',
                   edgecolor='white', linewidth=0.5)
ax.set_title(f'{experiment_name} — NoVS per Problem (α={level_significance})', fontsize=12)
ax.set_xlabel('Problem')
ax.set_ylabel('Number of Victories')
ax.legend(title='Solver', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.3)

novs_path = f'Figures/{experiment_name}/{experiment_name}_NoVS_per_problem.png'
fig.savefig(novs_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {novs_path}')

In [ ]:
# ── Lưu XLSX & LaTeX ─────────────────────────────────────────

xlsx_path = f'XLSX Files/{experiment_name}/{experiment_name}_NOVS.xlsx'
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    novs_cnt.to_frame().to_excel(writer, sheet_name='NoVS_total')
    ner_mean.to_excel(writer, sheet_name='NER_mean', index=False)
    ettq_mean.to_excel(writer, sheet_name='EtTQ_mean', index=False)
    if not novs_per_prob.empty:
        novs_per_prob.to_excel(writer, sheet_name='NoVS_per_problem')
print(f'Saved XLSX: {xlsx_path}')

# LaTeX NoVS table
latex_path = f'LaTeX/{experiment_name}/AAA_{experiment_name}_NoVS.txt'
latex_str  = novs_cnt.to_frame().to_latex()
with open(latex_path, 'w', encoding='utf-8') as f:
    f.write(latex_str)
print(f'Saved LaTeX: {latex_path}')

# LaTeX NER table
ner_latex_path = f'LaTeX/{experiment_name}/AAA_{experiment_name}_NER.txt'
ner_pivot_latex = ner_mean.pivot(index='problem', columns='solver', values='mean_NER').round(4)
with open(ner_latex_path, 'w', encoding='utf-8') as f:
    f.write(ner_pivot_latex.to_latex())
print(f'Saved LaTeX: {ner_latex_path}')

In [ ]:
# ── Summary table ─────────────────────────────────────────────
print('\n=== SUMMARY ===')
print(f'Experiment : {experiment_name}')
print(f'Problems   : {len(problems)}')
print(f'Solvers    : {len(solvers)}')
print(f'Total rows : {len(df_raw):,}')
print(f'Q3 threshold (EtTQ): {q3_threshold:.6f}')
print()
print('=== NoVS (tổng) ===')
print(novs_cnt.sort_values(ascending=False).to_string())
print()
print('=== Mean NER (trung bình qua tất cả problems) ===')
print(ner_mean.groupby('solver')['mean_NER'].mean().sort_values(ascending=False).round(4).to_string())